In [0]:
DECLARE catalog_name STRING DEFAULT 'bis_dev';
DECLARE bronze_schema STRING DEFAULT 'bronze_roster';
DECLARE silver_schema STRING DEFAULT 'silver_roster';

-- Create schema if it doesn't exist
EXECUTE IMMEDIATE
  'CREATE SCHEMA IF NOT EXISTS ' ||
  catalog_name || '.' || silver_schema;

-- Create materialized view
EXECUTE IMMEDIATE
'CREATE OR REPLACE MATERIALIZED VIEW ' || catalog_name || '.' || silver_schema || '.emp_storage_location
COMMENT ''Current storage location per representative. Most recent file only, one row per Emp_ID.''
AS
WITH latest AS (
    SELECT *
    FROM ' || catalog_name || '.' || bronze_schema || '.emp_storage_location
    WHERE File_Date = (
        SELECT MAX(File_Date)
        FROM ' || catalog_name || '.' || bronze_schema || '.emp_storage_location
    )
),
ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY Emp_ID
               ORDER BY Load_Timestamp DESC
           ) AS rn
    FROM latest
),
phone AS (
    SELECT *,
           regexp_replace(Rep_Phone, ''[^0-9]'', '''') AS digits
    FROM ranked
    WHERE rn = 1
)
SELECT
    Emp_ID,
    Rep_Name,
    Rep_Email,
    CASE
        WHEN length(digits) = 11 AND left(digits, 1) = ''1''
        THEN substr(digits, 2)
        ELSE digits
    END AS Rep_Phone,
    Territory_ID,
    Territory_Name,
    Region_Emp_Name,
    Region_Name,
    Facility_Name,
    Space_Number,
    Storage_Size,
    Facility_Raw_Address,
    Address_1,
    Address_2,
    City,
    State,
    Zip_Code,
    Parsing_Error,
    File_Date,
    File_Name
FROM phone';